# 03. Python 클래스로 센서 시뮬레이터 만들기

**모듈**: M1 - Python 기초 + 개발환경 세팅
**날짜**: 2026-03-09

실습 2에서 하드코딩한 센서 데이터를 **클래스**로 구조화합니다.

```
Sensor (기본 클래스)
├── CameraSensor     → 64x64x3 RGB
├── DistanceSensor   → 8방향 거리값
├── AudioSensor      → 2채널 스테레오
├── IMUSensor        → 6개 운동값
└── LLMSensor        → 텍스트 (상황해석 + 명령)

SensorManager → 5개 센서 통합 관리
```

**핵심 학습**: 클래스 = 데이터 + 동작을 하나로 묶는 설계도

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# macOS 한글 폰트 설정
mpl.rcParams['font.family'] = 'AppleGothic'
mpl.rcParams['axes.unicode_minus'] = False

np.random.seed(42)
print('라이브러리 로드 완료!')

## 1. Sensor 기본 클래스

모든 센서가 공통으로 가지는 것:
- `name`: 센서 이름
- `read()`: 데이터 읽기 (각 센서가 다르게 구현)
- `info()`: 센서 정보 출력

In [ ]:
class Sensor:
    """모든 센서의 기본 클래스"""
    
    def __init__(self, name, description):
        self.name = name
        self.description = description
    
    def read(self):
        """센서 데이터를 읽어서 반환 (하위 클래스에서 구현)"""
        raise NotImplementedError('하위 클래스에서 read()를 구현하세요')
    
    def info(self):
        """센서 정보 출력"""
        data = self.read()
        if isinstance(data, np.ndarray):
            print(f'[{self.name}] {self.description}')
            print(f'  shape: {data.shape}, dtype: {data.dtype}')
        else:
            print(f'[{self.name}] {self.description}')
            print(f'  type: {type(data).__name__}')

# 테스트: 기본 클래스 직접 호출하면 에러
try:
    s = Sensor('test', '테스트 센서')
    s.read()
except NotImplementedError as e:
    print(f'예상된 에러: {e}')
    print('→ 기본 클래스는 설계도일 뿐, 직접 사용 불가!')

## 2. CameraSensor - 로봇의 눈

64x64 RGB 이미지를 생성합니다.
- `obstacle_distance`: 장애물까지 거리 (0.0~1.0)
- 거리에 따라 장애물 크기가 변함 (가까우면 크게)

In [ ]:
class CameraSensor(Sensor):
    """64x64 RGB 카메라 센서"""
    
    def __init__(self):
        super().__init__('CAM', '64x64 RGB 카메라')
        self.width = 64
        self.height = 64
    
    def read(self, obstacle_distance=0.5):
        """카메라 이미지 생성
        
        Args:
            obstacle_distance: 장애물 거리 (0.0=매우 가까움, 1.0=멀리)
        Returns:
            np.ndarray: (64, 64, 3) RGB 이미지
        """
        img = np.zeros((self.height, self.width, 3), dtype=np.uint8)
        
        # 하늘 (위쪽 절반)
        img[:32, :, 2] = 180
        img[:32, :, 1] = 200
        
        # 바닥 (아래쪽 절반)
        img[32:, :, 1] = 150
        img[32:, :, 0] = 50
        
        # 장애물 (거리에 따라 크기 변화)
        if obstacle_distance < 0.8:
            size = int(10 + (1 - obstacle_distance) * 20)  # 가까울수록 큼
            cx, cy = 32, 32  # 중앙
            y1 = max(0, cy - size)
            y2 = min(64, cy + size)
            x1 = max(0, cx - size // 2)
            x2 = min(64, cx + size // 2)
            img[y1:y2, x1:x2, 0] = 255  # 빨간 장애물
            img[y1:y2, x1:x2, 1] = 0
            img[y1:y2, x1:x2, 2] = 0
        
        # 약간의 노이즈 추가 (현실감)
        noise = np.random.randint(-10, 10, img.shape, dtype=np.int16)
        img = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)
        
        return img

# 테스트
cam = CameraSensor()
cam.info()

# 거리별 카메라 이미지 비교
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for i, dist in enumerate([0.1, 0.3, 0.6, 0.9]):
    img = cam.read(obstacle_distance=dist)
    axes[i].imshow(img)
    axes[i].set_title(f'거리: {dist}')
    axes[i].axis('off')
fig.suptitle('[CAM] 장애물 거리에 따른 카메라 영상', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. DistanceSensor - 8방향 레이저

8방향(N, NE, E, SE, S, SW, W, NW) 거리를 측정합니다.
- `obstacle_direction`: 장애물이 있는 방향 (0=N, 1=NE, ...)
- 해당 방향은 가까운 값, 나머지는 먼 값

In [ ]:
class DistanceSensor(Sensor):
    """8방향 거리 센서"""
    
    DIRECTIONS = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']
    
    def __init__(self):
        super().__init__('DIST', '8방향 거리센서')
    
    def read(self, obstacle_direction=0, obstacle_distance=0.2):
        """거리 센서 데이터 생성
        
        Args:
            obstacle_direction: 장애물 방향 (0=N, 1=NE, ..., 7=NW)
            obstacle_distance: 장애물까지 거리 (0.0~1.0)
        Returns:
            np.ndarray: (8,) 각 방향별 거리값
        """
        # 기본: 모든 방향 먼 거리
        distances = np.ones(8) * 0.9 + np.random.uniform(-0.1, 0.1, 8)
        
        # 장애물 방향은 가까운 거리
        distances[obstacle_direction] = obstacle_distance
        # 인접 방향도 영향
        left = (obstacle_direction - 1) % 8
        right = (obstacle_direction + 1) % 8
        distances[left] = obstacle_distance + 0.2 + np.random.uniform(0, 0.1)
        distances[right] = obstacle_distance + 0.3 + np.random.uniform(0, 0.1)
        
        return np.clip(distances, 0.0, 1.0)

# 테스트
dist_sensor = DistanceSensor()
dist_sensor.info()

# 다양한 방향의 장애물
fig, axes = plt.subplots(1, 3, figsize=(15, 4), subplot_kw=dict(polar=True))
scenarios = [
    (0, 0.2, '앞(N)에 가까운 장애물'),
    (2, 0.3, '오른쪽(E)에 장애물'),
    (4, 0.1, '뒤(S)에 매우 가까운 장애물'),
]

for ax, (direction, distance, title) in zip(axes, scenarios):
    data = dist_sensor.read(obstacle_direction=direction, obstacle_distance=distance)
    angles = np.linspace(0, 2 * np.pi, 8, endpoint=False).tolist()
    angles += angles[:1]
    values = data.tolist() + [data[0]]
    
    ax.plot(angles, values, 'o-', linewidth=2, color='blue')
    ax.fill(angles, values, alpha=0.25, color='blue')
    ax.set_thetagrids([i * 45 for i in range(8)], DistanceSensor.DIRECTIONS)
    ax.set_ylim(0, 1.2)
    ax.set_title(title, pad=15, fontsize=10)

plt.suptitle('[DIST] 방향별 장애물 시나리오', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. AudioSensor - 스테레오 마이크

2채널 스테레오 오디오를 생성합니다.
- `sound_direction`: 소리 방향 (left/right/center)
- `sound_intensity`: 소리 세기 (0.0~1.0)

In [ ]:
class AudioSensor(Sensor):
    """2채널 스테레오 오디오 센서"""
    
    def __init__(self, sample_length=100):
        super().__init__('MIC', '2채널 스테레오 마이크')
        self.sample_length = sample_length
    
    def read(self, sound_direction='right', sound_intensity=0.5):
        """오디오 데이터 생성
        
        Args:
            sound_direction: 소리 방향 ('left', 'right', 'center')
            sound_intensity: 소리 세기 (0.0~1.0)
        Returns:
            np.ndarray: (2, sample_length) 스테레오 오디오
        """
        t = np.linspace(0, 8 * np.pi, self.sample_length)
        noise_l = np.random.normal(0, 0.05, self.sample_length)
        noise_r = np.random.normal(0, 0.05, self.sample_length)
        signal = sound_intensity * np.sin(t)
        
        if sound_direction == 'left':
            left = signal + noise_l
            right = noise_r
        elif sound_direction == 'right':
            left = noise_l
            right = signal + noise_r
        else:  # center
            left = signal * 0.7 + noise_l
            right = signal * 0.7 + noise_r
        
        return np.stack([left, right])

# 테스트
mic = AudioSensor()
mic.info()

# 방향별 소리 비교
fig, axes = plt.subplots(3, 1, figsize=(10, 6))
for ax, direction in zip(axes, ['left', 'center', 'right']):
    data = mic.read(sound_direction=direction, sound_intensity=0.5)
    ax.plot(data[0], color='blue', alpha=0.7, label='Left')
    ax.plot(data[1], color='red', alpha=0.7, label='Right')
    ax.set_title(f'소리 방향: {direction}', fontsize=10)
    ax.set_ylim(-1, 1)
    ax.legend(loc='upper right', fontsize=8)

plt.suptitle('[MIC] 방향별 오디오 신호', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. IMUSensor - 관성 측정 장치

로봇의 운동 상태 6개 값을 측정합니다.
- 속도: vx (전진), vy (횡방향)
- 가속도: ax, ay
- 회전: angular_velocity, angle

In [ ]:
class IMUSensor(Sensor):
    """6축 관성 측정 장치 (IMU)"""
    
    LABELS = ['vx', 'vy', 'ax', 'ay', 'angular_vel', 'angle']
    
    def __init__(self):
        super().__init__('IMU', '6축 관성센서')
    
    def read(self, speed=0.5, turn_rate=0.0):
        """IMU 데이터 생성
        
        Args:
            speed: 전진 속도 (0.0~1.0)
            turn_rate: 회전 속도 (-1.0=좌회전, 0.0=직진, 1.0=우회전)
        Returns:
            np.ndarray: (6,) IMU 값들
        """
        noise = np.random.normal(0, 0.02, 6)
        
        imu = np.array([
            speed,                          # vx: 전진 속도
            turn_rate * 0.1,                # vy: 횡방향 속도
            noise[2],                       # ax: 가속도 (거의 등속)
            turn_rate * 0.05,               # ay: 횡가속도
            turn_rate * 0.3,                # angular_velocity
            turn_rate * 15.0                # angle (도)
        ]) + noise
        
        return imu

# 테스트
imu_sensor = IMUSensor()
imu_sensor.info()

# 다양한 운동 상태
fig, axes = plt.subplots(1, 3, figsize=(14, 3))
scenarios = [
    (0.5, 0.0, '직진 중'),
    (0.3, 0.5, '우회전 중'),
    (0.0, 0.0, '정지'),
]

colors = ['#2196F3', '#2196F3', '#FF9800', '#FF9800', '#4CAF50', '#4CAF50']
for ax, (speed, turn, title) in zip(axes, scenarios):
    data = imu_sensor.read(speed=speed, turn_rate=turn)
    ax.barh(IMUSensor.LABELS, data, color=colors)
    ax.set_title(title, fontsize=10)
    ax.set_xlim(-1, 1)

plt.suptitle('[IMU] 운동 상태별 센서값', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. LLMSensor - 양방향 언어 센서

LLM은 다른 센서와 달리 **양방향**으로 동작합니다:
- **입력 (상황 해석)**: 다른 센서 데이터를 요약해서 의미를 부여
- **입력 (명령 수신)**: 사용자의 자연어 명령을 받음

지금은 텍스트 템플릿으로 구현하고, M5에서 Language Encoder가 벡터로 변환합니다.

In [ ]:
class LLMSensor(Sensor):
    """양방향 LLM 언어 센서"""
    
    def __init__(self):
        super().__init__('LLM', '양방향 언어센서 (해석+명령)')
    
    def read(self, sensor_summary=None, user_command=None):
        """LLM 센서 데이터 생성
        
        Args:
            sensor_summary: 다른 센서 요약 정보 (dict)
            user_command: 사용자 자연어 명령 (str)
        Returns:
            dict: {'interpretation': str, 'command': str}
        """
        # 상황 해석 생성 (다른 센서 정보 기반)
        if sensor_summary:
            interpretation = self._interpret(sensor_summary)
        else:
            interpretation = '센서 정보 없음'
        
        # 사용자 명령
        command = user_command or '명령 없음 (자율 주행 모드)'
        
        return {
            'interpretation': interpretation,
            'command': command
        }
    
    def _interpret(self, summary):
        """센서 요약을 자연어 해석으로 변환 (템플릿 기반)"""
        parts = []
        
        # 거리 센서 해석
        if 'closest_direction' in summary:
            d = summary['closest_direction']
            dist = summary['closest_distance']
            if dist < 0.3:
                parts.append(f'{d}방향 매우 가까운 거리에 장애물')
            elif dist < 0.6:
                parts.append(f'{d}방향에 장애물 감지')
        
        # 오디오 해석
        if 'sound_direction' in summary:
            sd = summary['sound_direction']
            if sd != 'none':
                dir_kr = {'left': '왼쪽', 'right': '오른쪽', 'center': '정면'}[sd]
                parts.append(f'{dir_kr}에서 소리 감지')
        
        # 운동 상태 해석
        if 'speed' in summary:
            if summary['speed'] > 0.3:
                parts.append('현재 전진 중')
            else:
                parts.append('현재 정지/저속')
        
        return '. '.join(parts) + '.' if parts else '특이사항 없음.'

# 테스트
llm = LLMSensor()
llm.info()

# 시나리오별 LLM 해석
scenarios = [
    (
        {'closest_direction': 'N', 'closest_distance': 0.2,
         'sound_direction': 'right', 'speed': 0.5},
        '오른쪽으로 돌아서 장애물을 피해'
    ),
    (
        {'closest_direction': 'E', 'closest_distance': 0.5,
         'sound_direction': 'left', 'speed': 0.3},
        None  # 자율 주행
    ),
    (
        {'closest_direction': 'S', 'closest_distance': 0.8,
         'sound_direction': 'none', 'speed': 0.0},
        '앞으로 출발해'
    ),
]

for i, (summary, cmd) in enumerate(scenarios, 1):
    result = llm.read(sensor_summary=summary, user_command=cmd)
    print(f'--- 시나리오 {i} ---')
    print(f'  해석: {result["interpretation"]}')
    print(f'  명령: {result["command"]}')
    print()

## 7. SensorManager - 5개 센서 통합 관리

`SensorManager`가 하는 일:
1. 5개 센서를 생성하고 관리
2. `read_all()` → 모든 센서 데이터를 한번에 반환
3. `dashboard()` → 통합 시각화

나중에 이 구조가 **Encoder → Fusion** 파이프라인의 입력이 됩니다.

In [ ]:
class SensorManager:
    """5개 센서를 통합 관리하는 매니저"""
    
    def __init__(self):
        self.camera = CameraSensor()
        self.distance = DistanceSensor()
        self.audio = AudioSensor()
        self.imu = IMUSensor()
        self.llm = LLMSensor()
        self.sensors = {
            'camera': self.camera,
            'distance': self.distance,
            'audio': self.audio,
            'imu': self.imu,
            'llm': self.llm,
        }
    
    def read_all(self, scenario=None):
        """모든 센서 데이터를 한번에 읽기
        
        Args:
            scenario: 시나리오 설정 dict (없으면 기본값)
        Returns:
            dict: 5개 센서 데이터
        """
        s = scenario or {}
        
        # 물리 센서 읽기
        obs_dist = s.get('obstacle_distance', 0.5)
        obs_dir = s.get('obstacle_direction', 0)
        sound_dir = s.get('sound_direction', 'right')
        sound_int = s.get('sound_intensity', 0.5)
        speed = s.get('speed', 0.5)
        turn = s.get('turn_rate', 0.0)
        command = s.get('user_command', None)
        
        camera_data = self.camera.read(obstacle_distance=obs_dist)
        distance_data = self.distance.read(obstacle_direction=obs_dir, obstacle_distance=obs_dist)
        audio_data = self.audio.read(sound_direction=sound_dir, sound_intensity=sound_int)
        imu_data = self.imu.read(speed=speed, turn_rate=turn)
        
        # 거리센서에서 가장 가까운 방향 찾기
        closest_idx = np.argmin(distance_data)
        sensor_summary = {
            'closest_direction': DistanceSensor.DIRECTIONS[closest_idx],
            'closest_distance': distance_data[closest_idx],
            'sound_direction': sound_dir,
            'speed': speed,
        }
        llm_data = self.llm.read(sensor_summary=sensor_summary, user_command=command)
        
        return {
            'camera': camera_data,
            'distance': distance_data,
            'audio': audio_data,
            'imu': imu_data,
            'llm': llm_data,
        }
    
    def dashboard(self, data, title='센서 대시보드'):
        """5개 센서 통합 시각화"""
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        fig.suptitle(title, fontsize=14, fontweight='bold')
        
        # 1. 카메라
        axes[0, 0].imshow(data['camera'])
        axes[0, 0].set_title('[CAM] 카메라 (64x64 RGB)')
        axes[0, 0].axis('off')
        
        # 2. 거리센서
        ax_radar = fig.add_subplot(2, 2, 2, polar=True)
        angles = np.linspace(0, 2 * np.pi, 8, endpoint=False).tolist()
        angles += angles[:1]
        values = data['distance'].tolist() + [data['distance'][0]]
        ax_radar.plot(angles, values, 'o-', linewidth=2, color='blue')
        ax_radar.fill(angles, values, alpha=0.25, color='blue')
        ax_radar.set_thetagrids([i * 45 for i in range(8)], DistanceSensor.DIRECTIONS)
        ax_radar.set_ylim(0, 1.2)
        ax_radar.set_title('[DIST] 거리센서 (8방향)', pad=20)
        axes[0, 1].set_visible(False)
        
        # 3. 오디오
        axes[1, 0].plot(data['audio'][0], color='blue', alpha=0.7, label='Left')
        axes[1, 0].plot(data['audio'][1], color='red', alpha=0.7, label='Right')
        axes[1, 0].set_title('[MIC] 오디오 (스테레오)')
        axes[1, 0].legend()
        axes[1, 0].set_ylim(-1, 1)
        
        # 4. IMU
        colors = ['#2196F3', '#2196F3', '#FF9800', '#FF9800', '#4CAF50', '#4CAF50']
        axes[1, 1].barh(IMUSensor.LABELS, data['imu'], color=colors)
        axes[1, 1].set_title('[IMU] 관성센서 (6개 값)')
        
        # 5. LLM 텍스트
        llm = data['llm']
        fig.text(0.5, 0.02,
                 f'[LLM] 해석: "{llm["interpretation"]}"\n         명령: "{llm["command"]}"',
                 ha='center', fontsize=10, style='italic',
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
        
        plt.tight_layout(rect=[0, 0.08, 1, 0.95])
        return fig

# SensorManager 생성
manager = SensorManager()
print(f'센서 매니저 생성 완료!')
print(f'관리 중인 센서: {list(manager.sensors.keys())}')

## 8. 시나리오별 테스트

`SensorManager.read_all()`로 다양한 상황을 시뮬레이션합니다.

In [ ]:
# 시나리오 1: 앞에 장애물 + 오른쪽에서 소리 (실습 2와 같은 상황)
scenario_1 = {
    'obstacle_distance': 0.2,
    'obstacle_direction': 0,      # N (앞)
    'sound_direction': 'right',
    'sound_intensity': 0.5,
    'speed': 0.5,
    'turn_rate': 0.0,
    'user_command': '오른쪽으로 돌아서 장애물을 피해',
}

data_1 = manager.read_all(scenario_1)

# 데이터 요약 출력
print('=== 시나리오 1: 정면 장애물 ===')
print(f'  camera:   {data_1["camera"].shape}')
print(f'  distance: {data_1["distance"].shape} → 최소: {data_1["distance"].min():.2f}')
print(f'  audio:    {data_1["audio"].shape}')
print(f'  imu:      {data_1["imu"].shape}')
print(f'  llm:      {data_1["llm"]["interpretation"]}')

fig = manager.dashboard(data_1, '시나리오 1: 정면에 장애물, 오른쪽에서 소리')
plt.show()

In [ ]:
# 시나리오 2: 옆에서 접근하는 물체 + 회피 중
scenario_2 = {
    'obstacle_distance': 0.4,
    'obstacle_direction': 2,      # E (오른쪽)
    'sound_direction': 'right',
    'sound_intensity': 0.7,
    'speed': 0.3,
    'turn_rate': -0.5,             # 왼쪽으로 회피
    'user_command': None,          # 자율 주행
}

data_2 = manager.read_all(scenario_2)

print('=== 시나리오 2: 오른쪽 장애물, 왼쪽 회피 ===')
print(f'  llm 해석: {data_2["llm"]["interpretation"]}')
print(f'  llm 명령: {data_2["llm"]["command"]}')

fig = manager.dashboard(data_2, '시나리오 2: 오른쪽 장애물, 왼쪽 회피 중')
plt.show()

In [ ]:
# 시나리오 3: 안전한 환경 (장애물 없음)
scenario_3 = {
    'obstacle_distance': 0.9,
    'obstacle_direction': 4,      # 뒤쪽 (신경 안 씀)
    'sound_direction': 'center',
    'sound_intensity': 0.1,       # 조용함
    'speed': 0.7,
    'turn_rate': 0.0,
    'user_command': '계속 직진해',
}

data_3 = manager.read_all(scenario_3)

print('=== 시나리오 3: 안전한 직진 ===')
print(f'  llm 해석: {data_3["llm"]["interpretation"]}')
print(f'  llm 명령: {data_3["llm"]["command"]}')

fig = manager.dashboard(data_3, '시나리오 3: 안전 - 고속 직진')
plt.savefig('../examples/sensor_dashboard_scenario3.png', dpi=100, bbox_inches='tight')
plt.show()

print('\n센서 대시보드 저장 완료: sensor_dashboard_scenario3.png')

## 9. 파이프라인 연결 미리보기

이 센서 데이터가 나중에 어떻게 사용되는지 정리합니다.

```
SensorManager.read_all()
    │
    ├── camera   (64,64,3) ──→ M5: VLM Encoder ──→ (16,)
    ├── distance (8,)      ──→ M5: Distance Enc ──→ (16,)
    ├── audio    (2,100)   ──→ M5: Audio Encoder ──→ (16,)
    ├── imu      (6,)      ──→ M5: Motion Encoder ──→ (16,)
    └── llm      {text}    ──→ M5: Language Enc  ──→ (16,)
                                      │
                               M6: 5-Modal Fusion (5×16 → 64)
                                      │
                               M7: V-JEPA World Model (미래 예측)
                                      │
                               M8: Flow Matching (행동 생성)
                                      │
                                    Action!
```

In [ ]:
# 파이프라인 데이터 흐름 확인
data = manager.read_all(scenario_1)

print('=== 파이프라인 데이터 흐름 ===')
print()
print('Step 1: SensorManager.read_all() 출력')
print(f'  camera:   {data["camera"].shape}  → {data["camera"].size:,}개 숫자')
print(f'  distance: {data["distance"].shape}      → {data["distance"].size}개 숫자')
print(f'  audio:    {data["audio"].shape}  → {data["audio"].size}개 숫자')
print(f'  imu:      {data["imu"].shape}      → {data["imu"].size}개 숫자')
print(f'  llm:      텍스트 2개     → M5에서 벡터화')
print()
print('Step 2: 각 Encoder → 16차원 벡터 (M5에서 구현)')
print('Step 3: 5-Modal Fusion → 64차원 통합 벡터 (M6에서 구현)')
print('Step 4: V-JEPA World Model → 미래 상태 예측 (M7에서 구현)')
print('Step 5: Flow Matching → 행동 시퀀스 생성 (M8에서 구현)')
print()
print('M1 완료! 센서 시뮬레이터가 전체 파이프라인의 첫 번째 블록입니다.')